In [5]:
import boto3
import pandas as pd
from io import StringIO
import os
from dotenv import load_dotenv
import numpy as np
import mlflow
import mlflow.sklearn
from datetime import datetime

load_dotenv()
# Use a local folder (writable) for MLflow artifacts
mlflow.set_tracking_uri("http://localhost:5001")

# MLflow Configuration

# mlflow.set_tracking_uri("http://localhost:5001")  # MLflow tracking server
mlflow.set_experiment("AQI_Weather_Prediction")


<Experiment: artifact_location='/mlflow/artifacts/1', creation_time=1765360583886, experiment_id='1', last_update_time=1765360583886, lifecycle_stage='active', name='AQI_Weather_Prediction', tags={'mlflow.experimentKind': 'custom_model_development'}>

In [ ]:
# Your AWS credentials and bucket info
bucket_name = 'my-feature-store-data'
s3_key = 'pipeline-data/data.csv'  # Example: "pipeline-data/data.csv"

# Create an S3 client
s3 = boto3.client(
    's3',
    aws_access_key_id= os.getenv('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=os.getenv('AWS_SECRET_ACCESS_KEY'),
)

# Fetch the object from S3
response = s3.get_object(Bucket=bucket_name, Key=s3_key)

# Read the CSV content
csv_data = response['Body'].read().decode('utf-8')

# Convert to DataFrame
df = pd.read_csv(StringIO(csv_data))

# Done!
print(df.isnull().sum())

index                         0
aqi_index                     0
co                            0
no                            0
no2                           0
o3                            0
so2                           0
pm2_5                         0
pm10                          0
nh3                           0
temperature_2m                0
relative_humidity_2m          0
precipitation                 0
wind_speed_10m                0
wind_direction_10m            0
surface_pressure              0
dew_point_2m                  0
apparent_temperature          0
shortwave_radiation           0
et0_fao_evapotranspiration    0
year                          0
month                         0
day                           0
hour                          0
Calculated_AQI                0
dtype: int64


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split


# Step 4: Define target and features
# Step 4: Define targets and features
target_columns = ["aqi_index", "Calculated_AQI"]
targets = df[target_columns]
X = df.drop(columns=target_columns)

# Step 5: Final check for datetime columns
X = X.select_dtypes(exclude=["datetime64[ns]"])

# Step 6: Split
X_train, X_test, y_train, y_test = train_test_split(X, targets, test_size=0.2, random_state=42)


In [ ]:
# Import necessary libraries
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split
import joblib
import numpy as np

In [ ]:
# Import additional required libraries
from io import BytesIO
from sklearn.multioutput import MultiOutputRegressor

# Step 3: Define Models with MultiOutput capability
models = {
    "Random_Forest": RandomForestRegressor(n_estimators=300, max_depth=10, random_state=42),
    "Gradient_Boosting": MultiOutputRegressor(GradientBoostingRegressor(n_estimators=300, max_depth=3, random_state=42)),
    "Linear_Regression": LinearRegression(),
    "Ridge_Regression": Ridge(alpha=1.0),
    "SVR": MultiOutputRegressor(SVR()),
    "Neural_Network": MultiOutputRegressor(MLPRegressor(max_iter=200, random_state=42))
}
artifact_root = os.path.abspath("./mlflow-artifacts")

# -----------------------
# Step 4: Train and Evaluate with MLflow Tracking
results = []
best_model = None
best_model_name = None
best_avg_rmse = float("inf")

for model_name, model in models.items():
    # Start MLflow run for each model
    with mlflow.start_run(run_name=f"{model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
        print(f"Training {model_name}...")
        
        # Log model parameters
        if hasattr(model, 'get_params'):
            mlflow.log_params(model.get_params())
        
        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        
        # Calculate metrics for each target
        model_results = {"Model": model_name}
        rmse_scores = []
        
        for i, col in enumerate(target_columns):
            rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], y_pred[:, i]))
            mae = mean_absolute_error(y_test.iloc[:, i], y_pred[:, i])
            r2 = r2_score(y_test.iloc[:, i], y_pred[:, i])
            
            rmse_scores.append(rmse)
            model_results[f"RMSE_{col}"] = rmse
            model_results[f"MAE_{col}"] = mae
            model_results[f"R²_{col}"] = r2
            
            # Log metrics to MLflow for each target
            mlflow.log_metric(f"rmse_{col}", rmse)
            mlflow.log_metric(f"mae_{col}", mae)
            mlflow.log_metric(f"r2_{col}", r2)
            
            print(f"  Target: {col}")
            print(f"    RMSE: {rmse}")
            print(f"    MAE: {mae}")
            print(f"    R²: {r2}")
        
        # Calculate average RMSE across all targets
        avg_rmse = np.mean(rmse_scores)
        avg_mae = np.mean([model_results[f"MAE_{col}"] for col in target_columns])
        avg_r2 = np.mean([model_results[f"R²_{col}"] for col in target_columns])
        
        model_results["Avg_RMSE"] = avg_rmse
        results.append(model_results)
        
        # Log aggregate metrics
        mlflow.log_metric("avg_rmse", avg_rmse)
        mlflow.log_metric("avg_mae", avg_mae)
        mlflow.log_metric("avg_r2", avg_r2)
        
        # Log model
        mlflow.sklearn.log_model(model)


        
        # Add tags
        mlflow.set_tag("model_type", model_name)
        mlflow.set_tag("targets", str(target_columns))
        
        print(f"  Average RMSE: {avg_rmse}\n")
        
        if avg_rmse < best_avg_rmse:
            best_avg_rmse = avg_rmse
            best_model = model
            best_model_name = model_name

# -----------------------
# Step 5: Register Best Model in MLflow Model Registry
print(f"\nBest model: {best_model_name} (Average RMSE = {best_avg_rmse:.2f})")

# Log and register the best model
with mlflow.start_run(run_name=f"BEST_{best_model_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"):
    mlflow.sklearn.log_model(
        best_model, 
        "best_model",
        registered_model_name="AQI_Weather_Best_Model"
    )
    mlflow.log_metric("best_avg_rmse", best_avg_rmse)
    mlflow.set_tag("best_model", best_model_name)
    mlflow.set_tag("production_ready", "true")

# -----------------------
# Step 6: Summary
results_df = pd.DataFrame(results)
print("\nSummary of Model Performance:")
print(results_df)


Training Random_Forest...


Training Random_Forest...


2025/12/10 15:45:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Training Random_Forest...


2025/12/10 15:45:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Target: aqi_index
    RMSE: 0.36500718109324537
    MAE: 0.2305995909304157
    R²: 0.8524611720320252
  Target: Calculated_AQI
    RMSE: 9.905600113371687
    MAE: 1.7958317816927203
    R²: 0.9895172138590425
🏃 View run Random_Forest_20251210_154502 at: http://localhost:5001/#/experiments/1/runs/7533086a700f4777a23ffc1cdb708e70
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run Random_Forest_20251210_154502 at: http://localhost:5001/#/experiments/1/runs/7533086a700f4777a23ffc1cdb708e70
🧪 View experiment at: http://localhost:5001/#/experiments/1


Training Random_Forest...


2025/12/10 15:45:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


  Target: aqi_index
    RMSE: 0.36500718109324537
    MAE: 0.2305995909304157
    R²: 0.8524611720320252
  Target: Calculated_AQI
    RMSE: 9.905600113371687
    MAE: 1.7958317816927203
    R²: 0.9895172138590425
🏃 View run Random_Forest_20251210_154502 at: http://localhost:5001/#/experiments/1/runs/7533086a700f4777a23ffc1cdb708e70
🧪 View experiment at: http://localhost:5001/#/experiments/1
🏃 View run Random_Forest_20251210_154502 at: http://localhost:5001/#/experiments/1/runs/7533086a700f4777a23ffc1cdb708e70
🧪 View experiment at: http://localhost:5001/#/experiments/1


OSError: [Errno 30] Read-only file system: '/mlflow'

In [ ]:
import joblib
import json
import sklearn
import numpy as np
from io import BytesIO

S3_MODEL_KEY = "models/best_model.pkl"
S3_METADATA_KEY = "models/best_model_metadata.json"

def upload_model_to_s3(model, bucket_name, s3_client):
    # --- Save model to BytesIO buffer ---
    model_buffer = BytesIO()
    joblib.dump(model, model_buffer)
    model_buffer.seek(0)
    s3_client.upload_fileobj(model_buffer, Bucket=bucket_name, Key=S3_MODEL_KEY)
    print(f"Model uploaded to s3://{bucket_name}/{S3_MODEL_KEY}")

    # --- Save version metadata ---
    metadata = {
        "sklearn_version": sklearn.__version__,
        "numpy_version": np.__version__,
        "model_type": type(model).__name__,
    }

    metadata_buffer = BytesIO()
    metadata_buffer.write(json.dumps(metadata).encode("utf-8"))
    metadata_buffer.seek(0)
    s3_client.upload_fileobj(metadata_buffer, Bucket=bucket_name, Key=S3_METADATA_KEY)
    print(f"Metadata uploaded to s3://{bucket_name}/{S3_METADATA_KEY}")


In [ ]:
upload_model_to_s3(best_model, bucket_name, s3)

Model uploaded to s3://my-feature-store-data/models/best_model.pkl
Metadata uploaded to s3://my-feature-store-data/models/best_model_metadata.json
